# InfraSight: YOLOv8n Fine-Tuning for Municipal Infrastructure Detection
### Project: InfraSight — Indore Municipal Corporation (IMC)
This Google Colab notebook fine-tunes **YOLOv8n** on the **Road Damage Dataset 2022 (RDD2022 - India subset)** and Roboflow pothole & garbage datasets, then exports `best.pt` for deployment into `backend/weights/best.pt`.

In [ ]:
# 1. Verify GPU Acceleration
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics roboflow pyyaml

In [ ]:
# 3. Create Dataset Configuration (data.yaml)
import yaml

data_config = {
    'path': './dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'pothole',
        1: 'damaged_road',
        2: 'broken_streetlight',
        3: 'overflowing_drain',
        4: 'garbage'
    }
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print('Dataset configuration saved to data.yaml')

In [ ]:
# 4. Download and Prepare Training Data
# Using Roboflow dataset downloader (Replace with your Roboflow API key if fine-tuning from scratch)
from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# project = rf.workspace("infrasight").project("indore-road-damage")
# dataset = project.version(1).download("yolov8")
print('Data download placeholder ready. You can also mount Google Drive or upload RDD2022 zip.')

In [ ]:
# 5. Fine-Tune YOLOv8n on Municipal Defect Classes
from ultralytics import YOLO

# Load pretrained nano model
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='infrasight_indore_v1',
    device=0,
    patience=10,
    save=True
)

In [ ]:
# 6. Evaluate Model Performance (mAP50-95)
metrics = model.val()
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# 7. Download Exported best.pt to local InfraSight backend
from google.colab import files
files.download('runs/detect/infrasight_indore_v1/weights/best.pt')